# 🤖 Project 12.1 — Robot Step Cost Planner
**DSA for Mechatronics · Week 12 — Dynamic Programming**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, time
from functools import lru_cache
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A robot navigates a sensor-lined corridor. Each step has an energy cost.
We solve three 1D DP problems on your personalised corridor data:

1. **Min-cost stair climber** — reach the top spending as little energy as possible
   (can climb 1 or 2 steps; start from step 0 or 1)
2. **Max-harvest robot** — collect maximum sensor readings without triggering
   two adjacent sensors (House Robber pattern)
3. **Peak-energy window** — find the contiguous segment with the highest total energy
   (Kadane's algorithm — maximum subarray)

For problem 1 we implement **both** memoization and tabulation and compare them.


## Step 1 — Generate corridor dataset

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_STEPS     = random.randint(10, 18)
N_SENSORS   = random.randint(12, 20)
N_ENERGY    = random.randint(15, 25)

STEP_COSTS  = [random.randint(1, 30) for _ in range(N_STEPS)]
SENSOR_VALS = [random.randint(5, 80) for _ in range(N_SENSORS)]
ENERGY_SEQ  = [random.randint(-20, 30) for _ in range(N_ENERGY)]

print(f"Corridor parameters:")
print(f"  Step costs  (n={N_STEPS}) : {STEP_COSTS}")
print(f"  Sensor vals (n={N_SENSORS}) : {SENSOR_VALS}")
print(f"  Energy seq  (n={N_ENERGY}) : {ENERGY_SEQ}")


## Step 2 — Min-cost stair climber (memoization + tabulation)

In [ ]:
# ── MEMOIZATION (top-down) ───────────────────────────────────────
def min_cost_memo(cost):
    """
    Top-down DP with memoization.

    State: min_cost(i) = minimum energy to reach step i.
    Recurrence: min_cost(i) = cost[i] + min(min_cost(i-1), min_cost(i-2))
    Base cases: min_cost(0) = cost[0],  min_cost(1) = cost[1]
    Answer: min(min_cost(n-1), min_cost(n-2))  — can finish from last or second-last step.

    Top-down: recurse from the answer, cache results to avoid recomputation.
    Time O(n), Space O(n) for cache + call stack.
    """
    n = len(cost)
    memo = {}
    calls = [0]
    def dp(i):
        calls[0] += 1
        if i in memo: return memo[i]
        if i == 0: return cost[0]
        if i == 1: return cost[1]
        memo[i] = cost[i] + min(dp(i-1), dp(i-2))
        return memo[i]
    answer = min(dp(n-1), dp(n-2))
    return answer, calls[0], memo

# ── TABULATION (bottom-up) ───────────────────────────────────────
def min_cost_tab(cost):
    """
    Bottom-up DP with tabulation.

    Build the dp table iteratively from the base cases upward.
    Same recurrence, no recursion, no call-stack overhead.
    Time O(n), Space O(n) for table.
    Can be space-optimised to O(1) by keeping only last two values.
    """
    n = len(cost)
    dp = [0] * n
    dp[0] = cost[0]
    dp[1] = cost[1]
    for i in range(2, n):
        dp[i] = cost[i] + min(dp[i-1], dp[i-2])
    return min(dp[-1], dp[-2]), dp

# ── SPACE-OPTIMISED (O(1) space) ─────────────────────────────────
def min_cost_opt(cost):
    """Space-optimised: only keep previous two values."""
    n = len(cost)
    if n == 1: return cost[0]
    prev2, prev1 = cost[0], cost[1]
    for i in range(2, n):
        curr = cost[i] + min(prev1, prev2)
        prev2, prev1 = prev1, curr
    return min(prev1, prev2)

memo_ans, memo_calls, memo_cache = min_cost_memo(STEP_COSTS)
tab_ans,  dp_table              = min_cost_tab(STEP_COSTS)
opt_ans                         = min_cost_opt(STEP_COSTS)

print(f"Min-cost stair climber (n={N_STEPS} steps):")
print(f"  Step costs : {STEP_COSTS}")
print()
print(f"  Memoization result  : {memo_ans}  ({memo_calls} function calls)")
print(f"  Tabulation result   : {tab_ans}")
print(f"  Space-opt result    : {opt_ans}")
print(f"  All agree           : {'✅ YES' if memo_ans == tab_ans == opt_ans else '❌ NO'}")
print()
print(f"  DP table (tabulation):")
print(f"    index : {list(range(N_STEPS))}")
print(f"    cost  : {STEP_COSTS}")
print(f"    dp    : {dp_table}")
print()

# Reconstruct optimal path (backtrack from best end)
path = []
i = len(dp_table) - 1 if dp_table[-1] < dp_table[-2] else len(dp_table) - 2
path.append(i)
while i > 1:
    i = i - 1 if dp_table[i-1] <= dp_table[i-2] else i - 2
    path.append(i)
path.reverse()
print(f"  Optimal steps taken : {path}")
print(f"  Costs on path       : {[STEP_COSTS[i] for i in path]}")
print(f"  Path total          : {sum(STEP_COSTS[i] for i in path)} == {memo_ans}")


## Step 3 — Max-harvest robot (House Robber)

In [ ]:
def house_robber(nums):
    """
    House Robber: collect maximum from non-adjacent elements.

    State: dp[i] = max collectible from sensors 0..i.
    Recurrence:
      dp[i] = max(dp[i-1],          # skip sensor i
                  dp[i-2] + nums[i]) # collect sensor i (skip i-1)
    Base: dp[0] = nums[0],  dp[1] = max(nums[0], nums[1])
    Answer: dp[n-1]

    Space-optimised version: keep only prev2, prev1.
    O(n) time, O(1) space.
    """
    n = len(nums)
    if n == 0: return 0, []
    if n == 1: return nums[0], [0]
    dp = [0] * n
    dp[0] = nums[0]
    dp[1] = max(nums[0], nums[1])
    for i in range(2, n):
        dp[i] = max(dp[i-1], dp[i-2] + nums[i])
    # Backtrack to find which sensors were collected
    collected = []
    i = n - 1
    while i >= 0:
        if i == 0 or dp[i] != dp[i-1]:
            collected.append(i)
            i -= 2
        else:
            i -= 1
    collected.reverse()
    return dp[-1], collected, dp

robber_max, collected_idx, rob_dp = house_robber(SENSOR_VALS)

print(f"Max-harvest robot — House Robber (n={N_SENSORS} sensors):")
print(f"  Sensor values : {SENSOR_VALS}")
print()
print(f"  DP table : {rob_dp}")
print()
print(f"  Max collectible     : {robber_max}")
print(f"  Sensors collected   : indices {collected_idx}")
print(f"  Values collected    : {[SENSOR_VALS[i] for i in collected_idx]}")
print(f"  Sum of collected    : {sum(SENSOR_VALS[i] for i in collected_idx)}")
# Verify no two adjacent
adjacent_ok = all(collected_idx[i+1] - collected_idx[i] >= 2
                  for i in range(len(collected_idx)-1))
print(f"  No adjacent selected: {'✅ YES' if adjacent_ok else '❌ NO'}")


## Step 4 — Peak-energy window (Kadane's algorithm)

In [ ]:
def kadane(nums):
    """
    Maximum subarray sum (Kadane's algorithm).

    Key idea: at each position, decide whether to extend the current subarray
    or start a new one from scratch:
      current = max(nums[i], current + nums[i])
                    ↑ start fresh    ↑ extend

    This is a 1D DP where dp[i] = max subarray ending at i.
    We just keep a running max instead of a full table.
    O(n) time, O(1) space.

    Handles all-negative arrays (returns the single least-negative element).
    """
    max_sum  = nums[0]
    curr_sum = nums[0]
    start = end = best_start = 0
    for i in range(1, len(nums)):
        if curr_sum + nums[i] < nums[i]:
            curr_sum = nums[i]
            start = i
        else:
            curr_sum += nums[i]
        if curr_sum > max_sum:
            max_sum   = curr_sum
            best_start = start
            end       = i
    return max_sum, best_start, end

kd_sum, kd_start, kd_end = kadane(ENERGY_SEQ)

print(f"Peak-energy window — Kadane's algorithm (n={N_ENERGY}):")
print(f"  Energy sequence : {ENERGY_SEQ}")
print()
print(f"  Max subarray sum  : {kd_sum}")
print(f"  Window indices    : [{kd_start}, {kd_end}]  (inclusive)")
print(f"  Window values     : {ENERGY_SEQ[kd_start:kd_end+1]}")
print(f"  Window sum verify : {sum(ENERGY_SEQ[kd_start:kd_end+1])}")
print(f"  Correct           : {'✅ YES' if sum(ENERGY_SEQ[kd_start:kd_end+1]) == kd_sum else '❌ NO'}")
print()

# Show step-by-step Kadane trace
print(f"  Kadane trace (first 10 steps):")
curr = ENERGY_SEQ[0]; mx = ENERGY_SEQ[0]; s = 0
print(f"    i=0 : curr={curr:4d}  max_so_far={mx:4d}  window=[0,0]")
for i in range(1, min(10, N_ENERGY)):
    extend = curr + ENERGY_SEQ[i]
    fresh  = ENERGY_SEQ[i]
    if fresh > extend:
        curr = fresh; s = i
    else:
        curr = extend
    if curr > mx: mx = curr
    print(f"    i={i} : curr={curr:4d}  max_so_far={mx:4d}  "
          f"{'(extend)' if fresh <= extend else '(restart)'}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " ROBOT STEP COST PLANNER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  MIN-COST STAIR CLIMBER" + " "*(W-24) + "║")
print(f"║  {'Steps (n)':<28}: {N_STEPS:<26}║")
print(f"║  {'Min cost (memo)':<28}: {memo_ans:<26}║")
print(f"║  {'Min cost (tabulation)':<28}: {tab_ans:<26}║")
print(f"║  {'Min cost (space-opt)':<28}: {opt_ans:<26}║")
print(f"║  {'Memo function calls':<28}: {memo_calls:<26}║")
print(f"║  {'Optimal path':<28}: {str(path):<26}║")
print("╠" + "═"*W + "╣")
print("║  MAX-HARVEST ROBOT (HOUSE ROBBER)" + " "*(W-34) + "║")
print(f"║  {'Sensors (n)':<28}: {N_SENSORS:<26}║")
print(f"║  {'Max collectible':<28}: {robber_max:<26}║")
print(f"║  {'Sensors collected':<28}: {len(collected_idx):<26}║")
print(f"║  {'No adjacent constraint':<28}: {'YES ✅' if adjacent_ok else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  PEAK-ENERGY WINDOW (KADANE)" + " "*(W-29) + "║")
print(f"║  {'Sequence length (n)':<28}: {N_ENERGY:<26}║")
print(f"║  {'Max subarray sum':<28}: {kd_sum:<26}║")
print(f"║  {'Window':<28}: [{kd_start}, {kd_end}]  len={kd_end-kd_start+1:<14}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about DP in this context?

*Your answer here:*

---

**Q2 — Memoization vs Tabulation — which did you find clearer, and why?**
Describe the trade-offs between top-down (memoization) and bottom-up (tabulation) for one of the problems in this project.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
